In [13]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [41]:
%cd data
!curl -L -O https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz


/Users/jaimesolis/de-zoomcamp/de-zoomcamp/06-batch/notebooks/data
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  123M  100  123M    0     0  1872k      0  0:01:07  0:01:07 --:--:--  870k


In [42]:
!gzip -d fhvhv_tripdata_2021-01.csv.gz

In [43]:
df= spark.read \
    .option("header", "true")\
    .csv("data/fhvhv_tripdata_2021-01.csv")

In [44]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [45]:
df.printSchema()


root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [46]:
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType)

schema = StructType([
    StructField("hvfhs_license_num", StringType(), True),
    StructField("dispatching_base_num", StringType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropoff_datetime", TimestampType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("SR_Flag", StringType(), True),
])

In [48]:
df = spark.read \
     .schema(schema) \
     .option("header", "true") \
    .csv("data/fhvhv_tripdata_2021-01.csv") 

In [49]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [50]:
df = df.repartition(24)

In [51]:
df.write.parquet("data/fhvhv/2021/01/")

26/02/26 22:29:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/26 22:29:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/26 22:29:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/26 22:29:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/26 22:29:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/26 22:29:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                      

In [52]:
df = spark.read.parquet('data/fhvhv/2021/01/')

In [53]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [54]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-04 19:26:08|2021-01-04 19:32:11|          35|          72|   NULL|
|           HV0003|              B02872|2021-01-02 17:23:32|2021-01-02 17:45:22|          73|         129|   NULL|
|           HV0005|              B02510|2021-01-03 01:26:11|2021-01-03 01:31:32|         116|         166|   NULL|
|           HV0005|              B02510|2021-01-03 15:37:11|2021-01-03 15:49:52|         244|         119|   NULL|
|           HV0005|              B02510|2021-01-01 16:42:14|2021-01-01 16:49:00|         181|         181|   NULL|
|           HV0003|              B02875|2021-01-03 18:48:12|2021-01-03 19:04:27|

In [55]:
from pyspark.sql import functions as F

def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'
        
crazy_stuff_udf = F.udf(crazy_stuff, returnType=StringType())

In [56]:
df.withColumn("pickup_date", F.to_date(df.pickup_datetime)) \
  .withColumn("dropoff_date", F.to_date(df.pickup_datetime)) \
  .withColumn("base_id", crazy_stuff_udf(df.dispatching_base_num))\
  .select("base_id", "pickup_date", "dropoff_date", "PULocationID", "DOLocationID") \
  .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-04|  2021-01-04|          35|          72|
|  e/b38| 2021-01-02|  2021-01-02|          73|         129|
|  e/9ce| 2021-01-03|  2021-01-03|         116|         166|
|  e/9ce| 2021-01-03|  2021-01-03|         244|         119|
|  e/9ce| 2021-01-01|  2021-01-01|         181|         181|
|  e/b3b| 2021-01-03|  2021-01-03|         142|         146|
|  a/a7a| 2021-01-02|  2021-01-02|         250|         213|
|  e/b3c| 2021-01-02|  2021-01-02|         255|         237|
|  e/b3c| 2021-01-02|  2021-01-02|         205|         205|
|  e/b35| 2021-01-05|  2021-01-05|          13|         146|
|  e/b38| 2021-01-04|  2021-01-04|         236|         170|
|  e/b38| 2021-01-01|  2021-01-01|          48|          68|
|  e/b14| 2021-01-04|  2021-01-04|         237|         236|
|  e/b3f| 2021-01-01|  2

In [57]:
df.filter(df.hvfhs_license_num == "HV0003") \
  .withColumn("base_id", crazy_stuff_udf(df.dispatching_base_num))\
  .select("base_id", "hvfhs_license_num", "pickup_datetime") \
  .show()

+-------+-----------------+-------------------+
|base_id|hvfhs_license_num|    pickup_datetime|
+-------+-----------------+-------------------+
|  e/b38|           HV0003|2021-01-02 17:23:32|
|  e/b3b|           HV0003|2021-01-03 18:48:12|
|  a/a7a|           HV0003|2021-01-02 14:03:28|
|  e/b3c|           HV0003|2021-01-02 15:47:37|
|  e/b3c|           HV0003|2021-01-02 14:59:29|
|  e/b35|           HV0003|2021-01-05 09:19:55|
|  e/b38|           HV0003|2021-01-04 14:39:39|
|  e/b38|           HV0003|2021-01-01 07:02:06|
|  e/b14|           HV0003|2021-01-04 20:48:43|
|  e/b3f|           HV0003|2021-01-01 17:47:15|
|  s/b36|           HV0003|2021-01-04 22:54:40|
|  e/b3b|           HV0003|2021-01-03 19:03:43|
|  e/b3f|           HV0003|2021-01-02 18:20:03|
|  s/acd|           HV0003|2021-01-02 17:55:34|
|  a/b40|           HV0003|2021-01-01 01:32:17|
|  s/acd|           HV0003|2021-01-05 00:14:07|
|  e/b3b|           HV0003|2021-01-02 21:44:13|
|  e/acc|           HV0003|2021-01-04 22

In [58]:
spark.stop()